# A Gentle Introduction to Spark on Databricks

Databricks is a cloud-first environment where Data Analysts, Data Scientists, and Data Engineers can collaborate and run **fully-managed Apache Spark** on the cloud.<br>

**Additional Features**: <br>
- Revision History and Version Control integration with: GitHub, Bitbucket, Azure DevOps
- Lots of different clusters/compute instances you can create then easily turn on/off
- Job scheduling for no additional cost

## Learning Objectives
By the end of this lesson, you should be able to:
* Use DBUtils and some common functions
* Create the SparkSession and use it to interact with Spark
* PySpark SQL Functions and DataFrames and casting functions
* Use Pandas to create your own DataFrames

🕒 **Estimated Time For Completion:** 25 minutes


### Databricks File System (DBFS)
- Databricks comes with a convenient 'file system' called **DBFS**. Actually, this is just an abstraction of some object storage service (e.g. Amazon S3 or Azure Blob Storage)
- Therefore, for large production jobs, I would actually recommend reading/writing to **S3 or Blob Storage explicitly**. In order to have universal access and control over your data
- However, for small/quick tasks **DBFS** can be pretty convenient 

### Databricks Utilities (DBUtils)
- You can access **DBFS** via `dbutils` (comes pre-installed with all Databricks clusters)
- dbutils actually has many other useful features such as: **programmatically executing other notebooks** & **accessing/managing secrets**
- For more info go to [**DBUtils Official Documentation**](https://docs.databricks.com/dev-tools/databricks-utils.html)

In [0]:
dbutils.fs.ls("/") # use the FS module of dbutils to access DBFS


### Your entry point to Spark:

`SparkSession` (often named as the variable `spark`)

All Databricks runtimes automatically names the `SparkSession` this way already.

In this example we're using an example dataset containing 20k random Amazon reviews that we can play around with.

In [0]:
spark

In [0]:
dbutils.fs.ls("./databricks-datasets/amazon/data20K/")

In [0]:
df = spark.read.format("parquet").load("dbfs:/databricks-datasets/amazon/data20K")

In [0]:
df.show() # show the dataframe in a simple tabular format

In [0]:
display(df) # more advanced functionality where we can see row numbers, sort the data or add visualisations


### Don't be afraid of making lots of DataFrame variables!

- After all, they're just a bunch of Lazy Transformations (in DAGs)
- There's no real resource consumption! In fact, it can even be better for clarity/debugging

In [0]:
import pyspark.sql.functions as F

# let's add a new column: instead of 1, 2, 3, 4, 5 let's make it a percentage between 0 and 1
# e.g. 4/5 => 80%
brand_new_column = F.col("rating_percentage") / 2 # with Column expressions, you don't need a column to even exist yet!

df = df.withColumn("rating_percentage", (F.col("rating") / F.lit(5)) * F.lit(100))

display(df.withColumn("half_rating_percentage", brand_new_column))


#### Experiment with Casting Columns

In [0]:
df.schema

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.types import *

df_cast_test = df

df_cast_test = df_cast_test.withColumn("rating", F.col("rating").cast(IntegerType()))
df_cast_test = df_cast_test.withColumn("rating_percentage", (F.col("rating") / F.lit(5)) * F.lit(100))
df_cast_test = df_cast_test.withColumn("review_timestamp", F.col("review").cast(TimestampType())) # let's see what happens if we try an impossible cast -> what would you expect?

display(df_cast_test)


#### Is there another way to create or rename columns?

Yes! Using the `alias` method

In [0]:
df_select_test = df

any_arbitrary_expression = (F.col("rating") + F.lit(1)).alias("arbitrary_expr")

# to add a new column, put "*" followed by your next arguments
df_select_test = df_select_test.select("*", any_arbitrary_expression) # you can also rename columns with alias()

display(df_select_test)

In [0]:
display(df_select_test.select(["rating_percentage", "review"]))


### Can I make my own DataFrames??

Let's see how we can create DataFrames from scratch using some data we provide ourselves, too.

In [0]:
import pandas as pd
from datetime import datetime 

pandas_df = pd.DataFrame({"x": [1, 2, 3, 4], "y": ["A", "B", "C", "D"]})
pandas_df["t"] = [datetime.now()] * len(pandas_df)
pandas_df

In [0]:
spark_df = spark.createDataFrame(pandas_df)
spark_df = spark_df.withColumn("z", F.col("t").cast(LongType())) # convert to Epoch seconds
spark_df = spark_df.withColumn("t_recreated", F.col("z").cast(TimestampType()))

In [0]:
spark_df.collect()

💡 [Collect the data from a DataFrame](https://sparkbyexamples.com/pyspark/pyspark-collect/)

In [0]:
display(spark_df)

| Previous Topic                                                              | Next Module                                                        |
|-----------------------------------------------------------------------------|-------------------------------------------------------------------|
| <a href="$./2.3 Apache Spark Primer" target="_self">Apache Spark Primer</a> | <a href="$../Module 03 - Data Ingestion and Basic Transformations/3.0 Table of Contents" target="_self">Data Ingestion and Basic Transformations</a> |

&copy; 2024 Thoughtworks. All rights reserved.<br/>